In [1]:
!pip uninstall -y tensorflow
!uv pip install --no-deps --system --no-index --find-links='/kaggle/input/hengck23-submit-physionet/hengck23-submit-physionet/setup' 'connected-components-3d'

Found existing installation: tensorflow 2.18.0
Uninstalling tensorflow-2.18.0:
  Successfully uninstalled tensorflow-2.18.0
Using Python 3.11.13 environment at: /usr
Resolved 1 package in 22ms
Prepared 1 package in 107ms
Installed 1 package in 6ms
 + connected-components-3d==3.26.1


In [2]:
import sys
sys.path.append('/kaggle/input/hengck23-submit-physionet/hengck23-submit-physionet')

import os, gc, cv2, torch, psutil, timm
import numpy as np
import pandas as pd
import torchvision.transforms as T
from tqdm.auto import tqdm
from scipy.signal import savgol_filter
from pathlib import Path

# --- CUSTOM MODULES ---
from stage0_model import Net as Stage0Net
from stage0_common import *
from stage1_model import Net as Stage1Net
from stage1_common import *
from stage2_model import *
from stage2_common import *

# --- HYPERPARAMETERS & PRECISION CONSTANTS ---
device = torch.device("cuda:0")
mv_to_pixel = 78.74   # Standard 200 DPI scale
t0, t1 = 235, 4161    # Verified Golden Window
x0, x1, y0, y1 = 0, 2176, 0, 1696
ref_zeros = [703.5, 987.5, 1271.5, 1531.5]

# --- PRECISION UTILITIES ---

def extract_soft_argmax(heatmap, centers, window=25):
    """
    SUB-PIXEL EXTRACTION: Calculates vertical center of mass.
    This eliminates 1-pixel rounding errors that cap SNR at 18dB.
    """
    C, H, W = heatmap.shape
    series = np.zeros((C, W), dtype=np.float32)
    y_coords = np.arange(H, dtype=np.float32)
    for c in range(C):
        cy = int(centers[c])
        y_s, y_e = max(0, cy - window), min(H, cy + window)
        # Vertical probability slice around the expected lead
        slice_p = heatmap[c, y_s:y_e, :] 
        weights = y_coords[y_s:y_e].reshape(-1, 1)
        # Soft-Argmax: Sum(Y * Prob) / Sum(Prob)
        denom = np.sum(slice_p, axis=0) + 1e-9
        series[c] = np.sum(slice_p * weights, axis=0) / denom
    return series

# --- MODEL INITIALIZATION ---

class Net3(nn.Module):
    def __init__(self):
        super(Net3, self).__init__()
        self.encoder = timm.create_model('resnet34.a3_in1k', pretrained=False, in_chans=3, num_classes=0, global_pool='')
        self.decoder = MyCoordUnetDecoder(in_channel=512, skip_channel=[256, 128, 64, 0], out_channel=[128, 64, 32, 16], scale=[2, 2, 2, 2])
        self.pixel = nn.Conv2d(16, 4, 1)
    def forward(self, x):
        e = encode_with_resnet(self.encoder, x)
        last, _ = self.decoder(feature=e[-1], skip=e[:-1][::-1] + [None])
        return self.pixel(last)

s0_net = Stage0Net(pretrained=False).to(device)
s0_net = load_net(s0_net, '/kaggle/input/hengck23-submit-physionet/hengck23-submit-physionet/weight/stage0-last.checkpoint.pth').eval()
s1_net = Stage1Net(pretrained=False).to(device)
s1_net = load_net(s1_net, '/kaggle/input/hengck23-submit-physionet/hengck23-submit-physionet/weight/stage1-last.checkpoint.pth').eval()
s2_net = Net3().to(device)
s2_net.load_state_dict(torch.load("/kaggle/input/physio-seg-public/pytorch/net3_009_4200/1/iter_0004200.pt"))
s2_net.eval()

resize_s2 = T.Resize((1696, 4352), interpolation=T.InterpolationMode.BILINEAR)

# --- INCREMENTAL PROCESSING ---
test_dir = Path("/kaggle/input/physionet-ecg-image-digitization/test")
test = pd.read_csv("/kaggle/input/physionet-ecg-image-digitization/test.csv")
test['id'] = test['id'].astype(str)
test_id = test['id'].unique().tolist()
test_gb = test.groupby('id')

SUBMISSION_FILE = 'submission.csv'
header = True

for i, sample_id in enumerate(tqdm(test_id, desc="Processing")):
    current_batch = []
    try:
        # 1. In-Memory 3-Stage Pipeline
        img = cv2.cvtColor(cv2.imread(str(test_dir/f'{sample_id}.png')), cv2.COLOR_BGR2RGB)
        with torch.no_grad(), torch.amp.autocast('cuda'):
            # Stage 0 & 1: Alignment & Rectification
            b0 = image_to_batch(change_color(img)).to(device)
            out0 = s0_net(b0)
            rot, kp = output_to_predict(img, b0, out0)
            img_norm, _, _ = normalise_by_homography(rot, kp)
            
            b1 = {'image': torch.from_numpy(np.ascontiguousarray(img_norm.transpose(2,0,1))).unsqueeze(0).to(device)}
            out1 = s1_net(b1)
            grid, _ = output_to_predict(img_norm, b1, out1)
            img_rect = rectify_image(img_norm, grid)
            
            # Stage 2: Segmentation Heatmap
            df_sub = test_gb.get_group(sample_id)
            target_len = df_sub[df_sub['lead']=='II'].iloc[0].number_of_rows
            crop = (img_rect[y0:y1, x0:x1] / 255.0).transpose(2, 0, 1)
            batch2 = resize_s2(torch.from_numpy(np.ascontiguousarray(crop)).unsqueeze(0)).float().to(device)
            out2 = s2_net(batch2)
            # SIGMOID is required for soft-argmax probability distribution
            probs = torch.sigmoid(out2).cpu().numpy()[0]

        # 2. Precision Sub-Pixel Extraction
        s_raw = extract_soft_argmax(probs[:, :, t0:t1], ref_zeros)
        
        final_series = np.zeros((4, target_len), dtype=np.float32)
        for row_i in range(4):
            # Safe-Edge Dynamic Baseline (Match the paper DC offset)
            quiet_zone = np.concatenate([s_raw[row_i, :50], s_raw[row_i, -50:]])
            actual_zero = np.median(quiet_zone)
            
            lead_mv = (actual_zero - s_raw[row_i]) / mv_to_pixel
            # Window 3: Minimum cleaning to preserve R-peak fidelity
            lead_mv = savgol_filter(lead_mv, window_length=3, polyorder=2).astype(np.float32)
            
            if len(lead_mv) != target_len:
                final_series[row_i] = np.interp(np.linspace(0, 1, target_len), 
                                                np.linspace(0, 1, len(lead_mv)), 
                                                lead_mv).astype(np.float32)
            else:
                final_series[row_i] = lead_mv

        # 3. Lead Mapping & CSV Writing
        d_series = {'II': final_series[3]}
        names = [['I','aVR','V1','V4'], ['II','aVL','V2','V5'], ['III','aVF','V3','V6']]
        for r in range(3):
            splits = np.array_split(final_series[r], 4)
            for k, s in zip(names[r], splits): d_series[k] = s

        for _, row in df_sub.iterrows():
            val = d_series.get(row.lead, np.zeros(row.number_of_rows, dtype=np.float32))
            if len(val) != row.number_of_rows:
                val = np.interp(np.linspace(0, 1, row.number_of_rows), 
                                np.linspace(0, 1, len(val)), val).astype(np.float32)
            current_batch.append(pd.DataFrame({'id': [f'{sample_id}_{x}_{row.lead}' for x in range(row.number_of_rows)], 'value': val}))

    except Exception:
        df_sub = test_gb.get_group(sample_id)
        for _, row in df_sub.iterrows():
            current_batch.append(pd.DataFrame({'id': [f'{sample_id}_{x}_{row.lead}' for x in range(row.number_of_rows)], 'value': np.zeros(row.number_of_rows, dtype=np.float32)}))

    # --- MEMORY SAFETY: Append to Disk & Flush RAM ---
    if current_batch:
        batch_df = pd.concat(current_batch, axis=0, ignore_index=True)
        batch_df.to_csv(SUBMISSION_FILE, mode='a', index=False, header=header)
        header = False 
    
    del current_batch
    if (i + 1) % 50 == 0:
        gc.collect()
        torch.cuda.empty_cache()

print("Fidelity-Elite Submission Complete.")

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

THIS_DIR: /kaggle/input/hengck23-submit-physionet/hengck23-submit-physionet
REF_PT: (9, 2)
/kaggle/input/hengck23-submit-physionet/hengck23-submit-physionet
/kaggle/input/hengck23-submit-physionet/hengck23-submit-physionet
<All keys matched successfully>
<All keys matched successfully>


Processing:   0%|          | 0/2 [00:00<?, ?it/s]

Fidelity-Elite Submission Complete.
